## Практика: Играем с векторными представлениями слов (Word Embeddings) (всего 4 балла)

Сегодня мы поиграем с векторными представлениями слов (word embeddings): обучим свои собственные небольшие эмбеддинги, загрузим предобученную модель из `gensim model zoo` и используем её для визуализации текстовых корпусов.

Всё это мы будем делать на наборе данных для обучения эмбеддингов.

__Требования:__  `pip install --upgrade nltk gensim bokeh`, но только если вы запускаете ноутбук локально.

**Мы что-то сделаем на семинаре, но здесь будет и часть вашего домашнего задания!**

In [ ]:
# скачиваем данные:
!wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora.txt
# альтернативная ссылка для скачивания: https://yadi.sk/i/BPQrUu1NaTduEw

In [ ]:
import numpy as np

with open("./quora.txt", encoding="utf-8") as file:
    data = list(file)

data[50]

__Токенизация:__ типичный первый шаг в задачах NLP — это разделение исходных данных на слова.
Текст, с которым мы работаем, представлен в сыром виде: со знаками препинания и эмодзи, прикреплёнными к некоторым словам, поэтому простого `str.split` будет недостаточно.

Воспользуемся __`nltk`__ — библиотекой, которая решает многие задачи NLP, такие как токенизация, стемминг или разметка частей речи (part-of-speech tagging).

In [ ]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

In [ ]:
# ЗАДАНИЕ: приведите всё к нижнему регистру и извлеките токены с помощью токенизатора. 
# `data_tok` должен быть списком списков токенов для каждой строки в `data`.

data_tok = []  # ВАШ КОД

In [ ]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "пожалуйста, преобразуйте каждую строку в список токенов (строк)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "пожалуйста, преобразуйте каждую строку в список токенов (строк)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "пожалуйста, убедитесь, что вы привели данные к нижнему регистру"

In [ ]:
print([' '.join(row) for row in data_tok[:2]])

__Векторы слов (Word vectors):__ как говорится, есть много способов обучить векторные представления слов. Существуют Word2Vec и GloVe с разными целевыми функциями. Есть также fastText, который использует модели на уровне символов для обучения эмбеддингов.

Выбор огромен, так что давайте начнём с малого: __gensim__ — это ещё одна NLP-библиотека, в которой реализовано множество векторных моделей, включая word2vec.

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(data_tok, 
                 vector_size=32,      # размер вектора эмбеддинга
                 min_count=5,         # учитывать слова, которые встретились не менее 5 раз
                 window=5).wv         # определяем контекст как окно из 5 слов вокруг целевого слова

# Из документации gensim
# wv: Этот объект, по сути, содержит сопоставление между словами и их эмбеддингами.
# После обучения его можно использовать напрямую для запроса этих эмбеддингов различными способами.

In [ ]:
# теперь можно получать векторы слов!
model.get_vector('anything')

In [ ]:
# или напрямую запрашивать похожие слова. Поэкспериментируйте!
model.most_similar('bread')

### Использование предобученной модели

Заняло некоторое время, не так ли? А теперь представьте обучение полноразмерных (100-300 измерений) векторных представлений на гигабайтах текста: статьях из Википедии или постах из Твиттера.

К счастью, в наши дни вы можете получить предобученную модель эмбеддингов в 2 строки кода (без SMS, обещаем).

После первой загрузки (или если вы удалите её вручную), модель сохраняется в директории `~/gensim_data` или `%USER_PATH%/gensim_data`. Это можно проверить, установив параметр `return_path` в `True`.

In [ ]:
import gensim.downloader as api
model = api.load('glove-twitter-100')

In [ ]:
model.most_similar(positive=["coder", "money"], negative=["brain"])

```

```

```

```

```

```

```

```


# Визуализация данных с помощью векторных представлений слов (1 балл)

Один из способов проверить, насколько хороши наши векторы, — это нарисовать их. Проблема в том, что эти векторы находятся в пространстве размерностью 30+ и выше, а мы, люди, привыкли к 2-3 измерениям.

К счастью, мы, специалисты по машинному обучению, знаем о методах __понижения размерности__.

Давайте используем их, чтобы нарисовать 1000 самых частотных слов.

In [ ]:
words = model.index_to_key[:1000] 

print(words[::100])

In [ ]:
# для каждого слова вычислите его вектор с помощью модели
word_vectors = []  # ВАШ КОД

In [ ]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()

#### Линейная проекция: PCA

Самый простой линейный метод понижения размерности — это **М**етод **Г**лавных **К**омпонент (**P**rincipal **C**omponent **A**nalysis, PCA).

С геометрической точки зрения, PCA пытается найти оси, вдоль которых наблюдается наибольшая дисперсия. «Естественные» оси, если хотите.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">


Под капотом он пытается разложить матрицу «объект-признак» $X$ на две меньшие матрицы: $W$ и $\hat W$, минимизируя _среднеквадратичную ошибку_:

$$\|(X W) \hat{W} - X\|^2_2 \to_{W, \hat{W}} \min$$
- $X \in \mathbb{R}^{n \times m}$ - матрица объектов (**центрированная**);
- $W \in \mathbb{R}^{m \times d}$ - матрица прямого преобразования;
- $\hat{W} \in \mathbb{R}^{d \times m}$ - матрица обратного преобразования;
- $n$ объектов, $m$ исходных измерений и $d$ целевых измерений;



In [ ]:
from sklearn.decomposition import PCA

# спроецируйте векторы слов на 2D-плоскость с помощью PCA. Используйте старый-добрый API sklearn (fit, transform)
# после этого нормализуйте векторы, чтобы убедиться, что у них нулевое среднее и единичная дисперсия
word_vectors_pca = []  # ВАШ КОД

# и, возможно, ЕЩЁ ВАШ КОД здесь :)

In [ ]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "для каждого слова должен быть 2d-вектор"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "точки должны быть центрированы относительно нуля"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "точки должны иметь единичную дисперсию"

#### Давайте нарисуем!

In [ ]:
import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    """ рисует интерактивный график для точек данных с дополнительной информацией при наведении """
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show: pl.show(fig)
    return fig

In [ ]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

# наведите курсор на точки и посмотрите, сможете ли вы определить кластеры

#### Визуализация соседей с помощью t-SNE
PCA хорош, но он строго линеен и поэтому способен улавливать только грубую, высокоуровневую структуру данных.

Если же мы хотим сосредоточиться на сохранении близких соседей, мы можем использовать t-SNE, который сам по себе является методом вложения (эмбеддинга). Здесь вы можете прочитать __[подробнее о t-SNE](https://distill.pub/2016/misread-tsne/)__.

In [ ]:
from sklearn.manifold import TSNE

# спроецируйте векторы слов на 2D-плоскость с помощью t-SNE. подсказка: не паникуйте, обучение может занять минуту или две.
# нормализуйте их так же, как и с PCA


word_tsne = []  # ВАШ КОД

In [ ]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

### Визуализация фраз

Векторные представления слов также можно использовать для представления коротких фраз. Самый простой способ — взять __среднее__ векторов всех токенов во фразе.

Этот трюк полезен для понимания, с какими данными вы работаете: можно найти выбросы, кластеры или другие артефакты.

Давайте опробуем этот новый «молоток» на наших данных!

In [ ]:
def get_phrase_embedding(phrase):
    """
    Преобразует фразу в вектор путем агрегации эмбеддингов её слов. См. описание выше.
    """
    # 1. приведите фразу к нижнему регистру
    # 2. токенизируйте фразу
    # 3. усредните векторы слов для всех слов в токенизированной фразе
    # пропустите слова, которых нет в словаре модели
    # если все слова отсутствуют в словаре, верните нули
    
    vector = np.zeros([model.vector_size], dtype='float32')
    
    # ВАШ КОД
    
    return vector
        
    

In [ ]:
vector = get_phrase_embedding("I'm very sure. This never happened to me before...")

assert np.allclose(vector[::10],
                   np.array([ 0.31807372, -0.02558171,  0.0933293 , -0.1002182 , -1.0278689 ,
                             -0.16621883,  0.05083408,  0.17989802,  1.3701859 ,  0.08655966],
                              dtype=np.float32))
assert np.array_equal(get_phrase_embedding("thisisgibberish"), np.zeros([model.vector_size], dtype='float32')), "крайний случай, когда все слова отсутствуют, должен обрабатываться так, как описано в комментариях к функции"

In [ ]:
# для первого запуска рассмотрим только часть фраз.
chosen_phrases = data[::len(data) // 1000]

# вычислите векторы для выбранных фраз
phrase_vectors = []  # ВАШ КОД

In [ ]:
assert isinstance(phrase_vectors, np.ndarray) and np.isfinite(phrase_vectors).all()
assert phrase_vectors.shape == (len(chosen_phrases), model.vector_size)

In [ ]:
# спроецируйте векторы в 2D-пространство с помощью pca, tsne или другого метода на ваш выбор
# не забудьте нормализовать

phrase_vectors_2d = TSNE().fit_transform(phrase_vectors)

phrase_vectors_2d = (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0)) / phrase_vectors_2d.std(axis=0)

In [ ]:
draw_vectors(phrase_vectors_2d[:, 0], phrase_vectors_2d[:, 1],
             phrase=[phrase[:50] for phrase in chosen_phrases],
             radius=20,)

**Ниже есть продолжение! Прокрутите вниз, когда будете готовы.**
```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```


### Классификация текстов: Классификация запрещённых комментариев

![img](https://github.com/yandexdataschool/nlp_course/raw/master/resources/banhammer.jpg)

__В этом ноутбуке__ вы создадите алгоритм, который классифицирует комментарии в социальных сетях на обычные и токсичные.
Как и во многих реальных случаях, у вас есть только небольшой (10^3) набор данных с размеченными вручную примерами. Мы решим эту проблему, используя как классические методы NLP, так и подход на основе эмбеддингов.

In [ ]:
# если вы используете colab, скачайте данные:
# !wget https://raw.githubusercontent.com/yandexdataschool/nlp_course/refs/heads/2025/week01_embeddings/comments.tsv -O ./comments.tsv -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
data = pd.read_csv("comments.tsv", sep='\t')

texts = data['comment_text'].values
target = data['should_ban'].values
data[50::200]

In [ ]:
from sklearn.model_selection import train_test_split
texts_train, texts_test, y_train, y_test = train_test_split(texts, target, test_size=0.5, random_state=42)

__Примечание:__ в целом, хорошей практикой является разделение данных на обучающую (train) и тестовую (test) выборки до того, как с ними что-либо делать.

Это защищает вас от возможной утечки данных на этапе предварительной обработки. Например, если вы решите выбрать в качестве признаков слова, присутствующие в непристойных твитах, вы должны считать эти слова только по обучающей выборке. В противном случае ваш алгоритм сможет «сжульничать» при оценке.

### Предобработка и токенизация

Комментарии содержат «сырой» текст со знаками препинания, буквами в разном регистре и даже символами новой строки.

Чтобы упростить все дальнейшие шаги, мы разделим текст на токены, разделенные пробелами, с помощью одного из токенизаторов nltk.

In [ ]:
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer()
preprocess = lambda text: ' '.join(tokenizer.tokenize(text.lower()))

text = 'How to be a grown-up at work: replace "fuck you" with "Ok, great!".'
print("до:", text,)
print("после:", preprocess(text),)

In [ ]:
# задание: предобработайте каждый комментарий в train и test

texts_train = []  # ВАШ КОД
texts_test = []  # ВАШ КОД

In [ ]:
assert texts_train[5] ==  'who cares anymore . they attack with impunity .'
assert texts_test[89] == 'hey todds ! quick q ? why are you so gay'
assert len(texts_test) == len(y_test)

### Решение: «мешок слов» (bag of words) (1 балл)

![img](http://www.novuslight.com/uploads/n/BagofWords.jpg)

Один из традиционных подходов к такой задаче — использовать признаки, основанные на «мешке слов»:
1. составить словарь из часто встречающихся слов (использовать только обучающие данные)
2. для каждого обучающего примера подсчитать, сколько раз в нем встречается каждое слово из словаря.
3. считать это количество признаком для какого-либо классификатора

__Примечание:__ на практике вы можете вычислять такие признаки с помощью sklearn. Однако, пожалуйста, не делайте этого в текущем задании.
* `from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer`

In [ ]:
# задание: найдите до k самых частотных токенов в texts_train,
# отсортируйте их по количеству вхождений (от большего к меньшему)
k = 10000

# ВАШ КОД

bow_vocabulary = []  # ВАШ КОД

print('примеры признаков:', sorted(bow_vocabulary)[::100])

In [ ]:
def text_to_bow(text):
    """ преобразует текстовую строку в массив с количеством токенов. Используйте bow_vocabulary. """
    # ВАШ КОД
    
    return np.array([], 'float32')

In [ ]:
X_train_bow = np.stack(list(map(text_to_bow, texts_train)))
X_test_bow = np.stack(list(map(text_to_bow, texts_test)))

In [ ]:
k_max = len(set(' '.join(texts_train).split()))
assert X_train_bow.shape == (len(texts_train), min(k, k_max))
assert X_test_bow.shape == (len(texts_test), min(k, k_max))
assert np.all(X_train_bow[5:10].sum(-1) == np.array([len(s.split()) for s in  texts_train[5:10]]))
assert len(bow_vocabulary) <= min(k, k_max)
assert X_train_bow[6, bow_vocabulary.index('.')] == texts_train[6].split().count('.')

Мы будем использовать простую линейную модель: __Логистическую регрессию__.
Возможно, это не самая «модная» модель, но у её простоты есть свои преимущества: линейная модель невероятно быстра и требует меньше данных для обучения.
Давайте воспользуемся sklearn!

In [ ]:
from sklearn.linear_model import LogisticRegression
bow_model = None  # ВАШ КОД ЗДЕСЬ - обучите логистическую регрессию

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

for name, X, y, model in [
    ('train', X_train_bow, y_train, bow_model),
    ('test ', X_test_bow, y_test, bow_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

test_accuracy = np.mean(bow_model.predict(X_test_bow) == y_test)
print(f"Точность модели: {test_accuracy:.3f}")
assert test_accuracy > 0.77, "Подсказка: настройте параметр C, чтобы улучшить производительность"
print("Отличная работа!")

### Задание: реализуйте TF-IDF признаки (1 балл)

Не все слова одинаково полезны. Можно приоритизировать редкие слова и уменьшить вес таких слов, как «и»/«или», используя __TF-IDF признаки__. Эта аббревиатура расшифровывается как __term frequency/inverse document frequency__ (частота слова / обратная частота документа) и означает именно это:

$$ \text{feature}_i = \frac{\text{Count}(word_i \in x)}{\text{Total number of words in } x} \times \log\left(\frac{N}{\text{Count}(word_i \in D) + \alpha}\right) $$


, где x - это один текст, D - ваш набор данных (коллекция текстов), N - общее количество документов, а $\alpha$ - сглаживающий гиперпараметр (обычно 1).
А $Count(word_i \in D)$ - это количество документов, в которых встречается $word_i$.

Также может быть хорошей идеей нормализовать каждый образец данных после вычисления TF-IDF признаков.

__Ваше задание:__ реализуйте TF-IDF признаки, обучите модель и оцените ROC-кривую. Сравните результат с базовой моделью BagOfWords, созданной выше.

Пожалуйста, не используйте встроенные в sklearn/nltk векторизаторы TF-IDF в своём решении :) Однако вы всё ещё можете использовать их для отладки.

Прокрутите вниз, когда закончите с TF-IDF!
```

```

```

```

```

```

```

```

```

```

```

```

```

```

```


### Решение получше: векторные представления слов (1 балл)

Давайте попробуем другой подход: вместо подсчета частот слов, мы преобразуем все слова в предобученные векторные представления и усредним их, чтобы получить признаки для текста.

Это должно дать нам два ключевых преимущества: (1) теперь у нас 10^2 признаков вместо 10^4, и (2) наша модель может обобщаться на слова, которых нет в обучающем наборе данных.

Мы начнем со стандартного подхода с предобученными векторами слов. Однако вы также можете попробовать:
* обучить эмбеддинги с нуля на релевантных (неразмеченных) данных
* умножить векторы слов на их обратную частоту в наборе данных (аналогично tf-idf)
* конкатенировать несколько различных эмбеддингов
    * вызовите `gensim.downloader.info()['models'].keys()`, чтобы получить список доступных моделей
* кластеризовать слова по их векторам и попробовать «мешок id кластеров»

__Примечание:__ загрузка предобученной модели может занять некоторое время. Это отличная возможность, чтобы налить себе чашечку чая/кофе и взять печенье. Или посмотреть пару серий сериала, если у вас медленный интернет.

In [ ]:
import gensim.downloader 
embeddings = gensim.downloader.load("fasttext-wiki-news-subwords-300")

# Если у вас мало оперативной памяти или низкая скорость загрузки, используйте "glove-wiki-gigaword-100". В этом случае игнорируйте все дальнейшие `assert`.

In [ ]:
def vectorize_sum(comment):
    """
    реализуйте функцию, которая преобразует предварительно обработанный комментарий в сумму векторов его токенов
    """
    embedding_dim = embeddings.vectors.shape[1]
    features = np.zeros([embedding_dim], dtype='float32')
    
    # ВАШ КОД
    
    return features

assert np.allclose(
    vectorize_sum("who cares anymore . they attack with impunity .")[::70],
    np.array([ 0.0108616 ,  0.0261663 ,  0.13855131, -0.18510573, -0.46380025])
)

In [ ]:
X_train_wv = np.stack([vectorize_sum(text) for text in texts_train])
X_test_wv = np.stack([vectorize_sum(text) for text in texts_test])

In [ ]:
wv_model = LogisticRegression().fit(X_train_wv, y_train)

for name, X, y, model in [
    ('bow train', X_train_bow, y_train, bow_model),
    ('bow test ', X_test_bow, y_test, bow_model),
    ('vec train', X_train_wv, y_train, wv_model),
    ('vec test ', X_test_wv, y_test, wv_model)
]:
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))

plt.plot([0, 1], [0, 1], '--', color='black',)
plt.legend(fontsize='large')
plt.grid()

assert roc_auc_score(y_test, wv_model.predict_proba(X_test_wv)[:, 1]) > 0.92, "что-то не так с вашими признаками"

Если всё прошло успешно, вы только что смогли уменьшить количество ошибок классификации в два раза.
Этот трюк очень полезен при работе с небольшими наборами данных. Однако, если у вас сотни тысяч примеров, существует целый ряд других методов для этого. Мы доберемся до них во второй части.

**Хотите узнать больше?**
* Посмотрите, какие еще эмбеддинги есть в `model zoo`: `gensim.downloader.info()`
* Взгляните на [эмбеддинги FastText](https://github.com/facebookresearch/fastText)